# Hands-on Lab: Interactive Visual Analytics with Folium – Solved

Solutions for TASK 1 (launch sites), TASK 2 (success/failed markers) and TASK 3 (distances to proximities).

> The original lab runs in JupyterLite (`piplite`, `js.fetch`). This version works in regular Jupyter: `pip install folium pandas`.

In [ ]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon

In [ ]:
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
try:
    spacex_df = pd.read_csv(URL)
except Exception:
    spacex_df = pd.read_csv('spacex_launch_geo.csv')  # download the CSV manually if there is no internet

spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

## TASK 1: Mark all launch sites on a map

In [ ]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for _, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    # Circle with the site name as popup
    folium.Circle(coordinate, radius=1000, color='#d35400', fill=True) \
        .add_child(folium.Popup(row['Launch Site'])).add_to(site_map)
    # Text label with the site name
    folium.map.Marker(
        coordinate,
        icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                     html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row['Launch Site'])
    ).add_to(site_map)

site_map

**Are all launch sites in proximity to the Equator line?**

Not exactly. The Florida sites (CCAFS, KSC) are at about 28.5° N and VAFB in California at about 34.6° N, so none is right on the Equator. Florida is as far south as the continental US allows, and a lower latitude gives the rocket extra eastward speed from Earth's rotation, which saves fuel for orbits launched toward the east.

**Are all launch sites in very close proximity to the coast?**

Yes. All four sites are on the coast (Atlantic for Florida, Pacific for VAFB). Launches go over the ocean, so a failure or falling stages do not endanger populated areas.

## TASK 2: Mark the success/failed launches for each site

In [ ]:
spacex_df['marker_color'] = spacex_df['class'].apply(lambda c: 'green' if c == 1 else 'red')
spacex_df.tail(10)

In [ ]:
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)

for index, record in spacex_df.iterrows():
    marker = folium.Marker(
        [record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=f"{record['Launch Site']} – {'Success' if record['class'] == 1 else 'Failed'}"
    )
    marker_cluster.add_child(marker)

site_map

Zooming into each cluster shows which sites have mostly green markers (high success rate) and which have more red ones. In this dataset **KSC LC-39A** has the highest share of green markers, and **CCAFS LC-40** the lowest.

## TASK 3: Distances from a launch site to its proximities

In [ ]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(position='topright', separator=' Long: ', empty_string='NaN',
                               lng_first=False, num_digits=20, prefix='Lat:',
                               lat_formatter=formatter, lng_formatter=formatter)
site_map.add_child(mouse_position)
site_map

In [ ]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0  # approximate radius of earth in km
    lat1, lon1, lat2, lon2 = map(radians, (lat1, lon1, lat2, lon2))
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

def add_proximity(site_map, site_lat, site_lon, lat, lon, color='blue'):
    """Marker with the distance label + PolyLine from the launch site to the point."""
    distance = calculate_distance(site_lat, site_lon, lat, lon)
    folium.Marker(
        [lat, lon],
        icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                     html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance))
    ).add_to(site_map)
    folium.PolyLine(locations=[[site_lat, site_lon], [lat, lon]], weight=2, color=color).add_to(site_map)
    return distance

In [ ]:
# Launch site used for the analysis: CCAFS SLC-40
site = launch_sites_df[launch_sites_df['Launch Site'] == 'CCAFS SLC-40'].iloc[0]
site_lat, site_lon = site['Lat'], site['Long']

# Coordinates of the closest points (picked on the map with MousePosition – check/adjust them on your map)
proximities = {
    'Coastline': (28.56367, -80.57163),
    'Railway':   (28.57205, -80.58527),
    'Highway':   (28.56420, -80.57089),
    'City (Titusville)': (28.6122, -80.8076),
}

for name, (lat, lon) in proximities.items():
    d = add_proximity(site_map, site_lat, site_lon, lat, lon)
    print(f"{name:20s} {d:6.2f} km")

site_map

> **Important:** the coordinates of the railway, highway and city are approximate. In the lab you are expected to pick them yourself with the `MousePosition` tool (hover over the nearest railway/highway/coast/city and read the coordinates), so replace them with your own readings if they differ on your map.

**Are launch sites in close proximity to railways?** Yes, about 1 km away. Railways are used to bring heavy rocket parts and equipment.

**Are launch sites in close proximity to highways?** Yes, within about 1 km. Highways are needed for people and cargo logistics.

**Are launch sites in close proximity to the coastline?** Yes, under 1 km. Launches go over water, so debris and failed rockets fall into the ocean.

**Do launch sites keep certain distance away from cities?** Yes. The nearest city (Titusville) is more than 20 km away. This keeps people safe from explosions, noise and falling debris in case of a failure.